# Optimizing QAOA angles

This notebook runs a full p=1 QAOA angle optimization on a random Sherrington-Kirkpatrick
instance: first noiselessly, then with classical measurement noise added to the samples. It
then reads the logged trajectories back from the result store, adds a brute-force angle grid
computed from exact expectation values, and draws both trajectories over the grid landscape.

The sampling itself always runs on the local `aer` simulator. If IBM credentials are present
in the environment or in a local `.env`, the SABRE pass manager routes against a real
device's topology through an IBM provider; otherwise the notebook prints
`IBM credentials absent: using 'aer'` and takes both the backend and the pass manager from
`aer`, so no quantum-hardware account is needed.

**Writes**: optimization trajectories into the standardized result store
(`DEFAULT_STORAGE_DIRECTORY`; with that variable unset it is `output/` at the repository
root), and the interactive plots as HTML files into `../temp/`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import os
from dotenv import load_dotenv
from quapopt import ancillary_functions as anf
from quapopt.circuits import backend_utilities as bck_utils
os.makedirs('../temp', exist_ok=True)

#load env variables -- mainly to read IBM account data for qiskit
load_dotenv()

#Everything below runs without an IBM account; with credentials present, the SABRE pass
#manager routes against a real device's topology instead of the local simulator's.
HAVE_IBM = any(os.getenv(k) for k in ('IBM_TOKEN', 'IBM_CREDENTIALS_PATH', 'IBM_ACCOUNT_NAME'))

### Generate a random Hamiltonian instance

* `generate_random_hamiltonian` is the one-call shortcut [A01](../A_generating_hamiltonians/A01_generate_and_save_Hamiltonians.ipynb) introduces. A01 also shows the
  fine-grained `build_hamiltonian_generator` path, for when you want per-knob control.
* This notebook compares OPTIMIZATION TRAJECTORIES, noiseless against noisy, on one instance.
  It reports no approximation ratio, so it runs no classical solver.
* **What to change here**: `number_of_qubits` (10 keeps the p=1 landscape cheap to grid),
  `seed_cost_hamiltonian`, `hamiltonian_model` and the coefficient distribution.
* **Result**: `cost_hamiltonian`, the instance every run below optimizes.


In [ ]:
from quapopt.ancillary_functions.presets import generate_random_hamiltonian
from quapopt.data_analysis.data_handling import (CoefficientsType,
                                                 CoefficientsDistribution,
                                                 HamiltonianModels)

number_of_qubits = 10
seed_cost_hamiltonian = 1

# A random Sherrington-Kirkpatrick instance: all-to-all two-body couplings, no local fields.
cost_hamiltonian = generate_random_hamiltonian(hamiltonian_model=HamiltonianModels.SherringtonKirkpatrick,
                                               number_of_qubits=number_of_qubits,
                                               seed=seed_cost_hamiltonian,
                                               localities=(2,),
                                               coefficients_type=CoefficientsType.CONTINUOUS,
                                               coefficients_distribution=CoefficientsDistribution.Uniform,
                                               coefficients_distribution_properties={'low': -1,
                                                                                     'high': 1,
                                                                                     'step': 1})

print("Class description (cost):", cost_hamiltonian.hamiltonian_class_description)
print("Instance description (cost):", cost_hamiltonian.hamiltonian_instance_description)

#Nothing below reports an approximation ratio, so this notebook never solves the instance.
#A01 shows how an instance gets its known solutions written into the store.


### The result store, and the identifiers that key it

* Every optimization below writes one row per objective-function call into the standardized
  result store. `experiment_set_id` groups those rows; `experiment_folders_hierarchy` decides
  which folders they land in.
* A later cell reads the same rows back through a `ResultsLogger` built from the same
  identifiers, which is how the notebook checks that the logging round-trips.
* **What to change here**: the folder hierarchy, and `logging_level` --
  `LoggingLevel.NONE` turns the writes off entirely, `DETAILED` keeps the per-call rows.
* **Result**: `logger_kwargs_main` and `logging_level`, read by every runner below.


In [ ]:
from quapopt.optimization.QAOA.implementation.QAOARunnerSampler import QAOARunnerSampler
from quapopt.data_analysis.data_handling import LoggingLevel

#Specify some kwargs for logging
# (this is generally not required, but here we want to log the results in a specific folder)
#Experiment set id is used to identify results in the database later. (note: here 'database' means just a bunch of hierarchical folders, mainly with .csv files)
experiment_set_id = f"{anf.create_random_uuid()}"

logging_level = LoggingLevel.DETAILED

#The logger will create subfolders specified here
logger_kwargs_main = {'experiment_folders_hierarchy': ['SimulationResults',
                                                       'TutorialQAOAOptimization'],
                      'experiment_set_id': experiment_set_id,  #Used to group the experiments
                      }

print(logger_kwargs_main)



### The Qiskit backend and its pass manager

* The sampling runs on the local `aer` simulator. An IBM account is optional and is
  used for one thing only: routing the SABRE pass manager against a real device's coupling
  map. Without credentials the notebook prints `IBM credentials absent: using 'aer'` and
  routes against the simulator, which has all-to-all connectivity.
* **What to change here**: `sampler_backend` (`'Qiskit'` or `'python'` -- the
  next cell dispatches on it), and `backend_name`, which becomes a device name when you want
  to compile against real hardware.
* **Result**: `qiskit_backend`, `qiskit_pass_manager` and `qiskit_sampler`.


In [ ]:
sampler_backend = 'qiskit'

if sampler_backend.lower() == 'qiskit':
    from quapopt.optimization.QAOA import QubitMappingType

    #The sampling itself runs on the local 'aer' simulator either way. The provider is only
    #needed when you point the SABRE pass manager at a real device's topology.
    backend_name = 'aer'  #might change to actual backend for simulation for real device

    #Here we assume that IBM's account data is in the environment variables. This is read from .env file in the root of the repo.
    #You can also just set the variable values here.
    #NOTE: not all of those variables are needed for setting up the account. Please see the QiskitRuntimeService for details
    if HAVE_IBM:
        #get the provider for the IBM Qiskit backend; we need it to run on actual hardware
        #and to route against a real device's topology
        provider_ibm = bck_utils.get_qiskit_provider(
            credentials_path=os.getenv('IBM_CREDENTIALS_PATH'),
            instance_ibm=os.getenv('IBM_INSTANCE_NAME'),
            account_name=os.getenv('IBM_ACCOUNT_NAME'),
            token=os.getenv('IBM_TOKEN'),
            channel=os.getenv('IBM_CHANNEL'),
        )
    else:
        print("IBM credentials absent: using 'aer'")
        provider_ibm = None

    #get the backend and the SABRE pass manager that compiles circuits for it
    #This can be changed if desired
    qiskit_backend = bck_utils.get_qiskit_backend(backend_name=backend_name,
                                                  backend_kwargs=None,  #use defaults
                                                  qiskit_provider=provider_ibm)
    qiskit_pass_manager, pass_manager_kwargs = bck_utils.get_qiskit_pass_manager(
        qiskit_backend=qiskit_backend,
        qubit_mapping_type=QubitMappingType.sabre,
        pass_manager_kwargs=None,  #use defaults
    )
    print('backend:', qiskit_backend.name)

    qiskit_sampler = bck_utils.create_qiskit_sampler(qiskit_backend=qiskit_backend,
                                                     simulation=True,
                                                     qiskit_sampler_options=None,  #use defaults
                                                     session_ibm=None,  #add if desired
                                                     override_to_noiseless_simulation=True,
                                                     #remove default device's noise model
                                                     )

### Wrap the backend in a QAOA sampler

* `QAOARunnerSampler` is the layer that turns "a backend that runs circuits" into "a function
  from angles to an energy estimate". It holds the cost Hamiltonian, builds the ansatz once,
  and logs every call.
* `store_full_information_in_history=False` keeps only the main data -- energies, angles,
  indices -- in the run history. Set it to `True` only when you need the raw samples or the
  statevector of every call in memory. This can overflow memory for certain settings. Note that they can be saved to drive anyway uising logging.
* **What to change here**: the backend branch (`initialize_backend_qiskit` or
  `initialize_backend_python`) and `store_n_best_results`.
* **Result**: `qaoa_sampler`, ready to be handed to an optimizer.


In [ ]:
logger_kwargs_sampler = {**logger_kwargs_main,
                         **{'experiment_instance_id': f"IdealSampling{sampler_backend.upper()}"}}

numpy_rng_sampling = np.random.default_rng(seed=0)
#Set up the raw sampler with Hamiltonians as an input
qaoa_sampler = QAOARunnerSampler(
    hamiltonian_representations_cost=[cost_hamiltonian],
    hamiltonian_representations_phase=None,
    store_n_best_results=1,
    store_full_information_in_history=False,
    numpy_rng_sampling=numpy_rng_sampling,
    logger_kwargs=logger_kwargs_sampler,
    logging_level=logging_level
)

#Here we basically run the same thing as before but inside the class;
#so the ansatze are set up for possibly multiple cost Hamiltonians
if sampler_backend.lower() == 'qiskit':
    qaoa_sampler.initialize_backend_qiskit(simulation=True,
                                           qiskit_backend=qiskit_backend,
                                           noiseless_simulation=True,
                                           qiskit_pass_manager=qiskit_pass_manager,
                                           qaoa_depth=1,
                                           qubit_indices_physical=None,
                                           classical_indices=None)

elif sampler_backend.lower() == 'python':
    qaoa_sampler.initialize_backend_python(backend='auto')

else:
    raise NotImplementedError("Only qiskit and python are supported")

#print(qaoa_sampler._backends[0].ansatz_circuit)

### Wrapper for optimizers

* In general, we will want to use our QAOA samplers with some optimizers.
* We can run:
1. Optuna -- a hyperparameter optimization framework. It has some nice optimizers and visualization features.
2. Custom grid sampling - we can run a simple grid sampling to get a feel for the landscape.
3. Scipy -- a classical optimization library. It has many optimizers, and we can wrap any of them to work with our QAOA samplers.




### The classical optimizer

* The classical optimizer proposes angles; the sampler scores them. `ScipyOptimizerWrapped`
  wraps scipy methods -- COBYQA here -- and `basinhopping` restarts it from perturbed
  points to escape local minima.
* **What to change here**: `optimizer_name`, `parameters_bounds` (one pair per angle; p=1
  gives two), `basinhopping_kwargs['niter']` (more restarts, more calls) and
  `starting_point`.
* **Result**: `classical_optimizer` and `qaoa_optimizer`, the object the runs below drive.


In [ ]:
from quapopt.optimization.parameter_setting.variational.QAOAOptimizationRunner import QAOAOptimizationRunner
from quapopt.optimization.parameter_setting.variational.scipy_tools.ScipyOptimizerWrapped import ScipyOptimizerWrapped

#we can specify the details of classical optimizer here.
classical_optimizer = ScipyOptimizerWrapped(parameters_bounds=[(-np.pi, np.pi),
                                                               (-np.pi/2,np.pi/2)] ,
                                            optimizer_name='COBYQA',
                                            optimizer_kwargs=None,
                                            basinhopping=True,
                                            basinhopping_kwargs={'niter': 1},
                                            starting_point=[0.01] * 2
                                            )

qaoa_optimizer = QAOAOptimizationRunner(qaoa_runner=qaoa_sampler)

### Run 1 -- noiseless

* The first optimization. Each of `number_of_function_calls` calls draws
  `number_of_samples` shots, estimates the mean energy from them, and hands it back to the
  optimizer.
* **What to change here**: `number_of_function_calls` and `number_of_samples` -- together
  they set the whole cost of this cell -- and `optimizer_seed`.
* **Result**: `best_results_noiseless` (the best BITSTRING found, not the best mean energy)
  and `optimization_results_noiseless`, whose `trials_dataframe` holds one row per call. The
  same rows also go to the result store under `IdealSamplingQISKIT`. The cell prints the best
  bitstring, its energy and the best mean energy, and shows the data of the call that sampled
  that bitstring as a table.


In [ ]:
#Number of objective function calls
number_of_function_calls = 200
#Number of measurements to estimate the expectation values
number_of_samples = 10 ** 3

best_results_noiseless, optimization_results_noiseless = qaoa_optimizer.run_optimization(qaoa_depth=1,
                                                                                         number_of_function_calls=number_of_function_calls,
                                                                                         number_of_samples=number_of_samples,
                                                                                         classical_optimizer=classical_optimizer,
                                                                                         optimizer_seed=42,
                                                                                         #Note that we can add readout noise here
                                                                                         measurement_noise=None,
                                                                                         verbosity=0,
                                                                                         show_progress_bar=True)

#note: "best" result here means the one containing the best BITSTRING, not necessary the best expected value!
best_result_noiseless = best_results_noiseless[0]
best_energy_bts_noiseless = best_result_noiseless[0]
best_bts_noiseless = best_result_noiseless[1][0]
best_metadata_noiseless = best_result_noiseless[1][-1]

print("BEST ENERGY (BTS, NOISELESS):", best_energy_bts_noiseless)
print("BEST BTS (NOISELESS):", best_bts_noiseless)
print('BEST MEAN ENERGY (NOISELESS):', optimization_results_noiseless.best_value)
best_metadata_noiseless.to_dataframe_main()

### Run 2 -- with measurement noise

* We can add classical measurement noise to the simulation. Here we will emulate amplitude damping.
* `ClassicalMeasurementNoiseSampler` flips bits of every sample after the circuit. `p_01` is
  the probability of reading a 1 as a 0; with `p_10 = None` a 0 is never read as a 1.
* The runner's logger moves to a new experiment instance id, so the rows of Run 2 are stored
  apart from those of Run 1.
* **What to change here**: `p_01` and `p_10`.
* **Result**: `best_results_noisy` and `optimization_results_noisy`. The cell prints the best
  results of both runs, one above the other.


In [ ]:
from quapopt.circuits.noise.simulation.ClassicalMeasurementNoiseSampler import ClassicalMeasurementNoiseSampler, \
    MeasurementNoiseType

# Fully asymmetric noise -- equivalent to amplitude damping at the end of the circuit
p_01 = 0.5
p_10 = None
CMNS = ClassicalMeasurementNoiseSampler(noise_type=MeasurementNoiseType.TP_1q_identical,
                                        noise_description={'p_01': p_01,
                                                           'p_10': p_10})

logger_kwargs_sampler_noisy = {**logger_kwargs_main,
                               **{'experiment_instance_id': f"NoisySampling{sampler_backend.upper()}"}}
qaoa_optimizer.qaoa_runner.reinitialize_logger(**logger_kwargs_sampler_noisy)

best_results_noisy, optimization_results_noisy = qaoa_optimizer.run_optimization(qaoa_depth=1,
                                                                                 number_of_function_calls=number_of_function_calls,
                                                                                 number_of_samples=number_of_samples,
                                                                                 classical_optimizer=classical_optimizer,
                                                                                 optimizer_seed=42,
                                                                                 #Note that are adding noise here
                                                                                 measurement_noise=CMNS,
                                                                                 verbosity=0,
                                                                                 show_progress_bar=True
                                                                                 )

best_result_noisy = best_results_noisy[0]

best_energy_bts_noisy = best_result_noisy[0]
best_bts_noisy = best_result_noisy[1][0]
best_metadata_noisy = best_result_noisy[1][-1]

print("BEST ENERGY (BTS, NOISELESS):", best_energy_bts_noiseless)
print("BEST BTS (NOISELESS):", best_bts_noiseless)
print('BEST MEAN ENERGY (NOISELESS):', optimization_results_noiseless.best_value)
print("____________")
print("BEST ENERGY (BTS, NOISY):", best_energy_bts_noisy)
print("BEST BTS:", best_bts_noisy)
print('BEST MEAN ENERGY (NOISY):', optimization_results_noisy.best_value)

### Reading data

* In a moment, we will want to visualize data.
* We could take the data directly from the output of optimization, but here we will read it from database to test the logging:
1. We need to set up the logger for the results. (the optimizer does it automatically, but we are pretending we ran the optimization separately and now just reading the results).
2. The input requires some logging metadata, such as potentially the folder hierarchy and the main table name prefix so it can identify what data we want to read.
3. We can then read the data from the database and visualize it.

* **What to change here**: `experiment_instance_ids`, to read other runs of the same
  experiment set.
* **Result**: four tables, shown one after another. `df_noiseless` and `df_noisy` are read back
  from the store; `df_noiseless_direct` and `df_noisy_direct` are the optimizer's own
  `trials_dataframe`. Each pair holds one row per call of the same run.


In [ ]:
from quapopt.data_analysis.data_handling import ResultsLogger
from quapopt.data_analysis.data_handling import (STANDARD_NAMES_DATA_TYPES as SNDT,
                                                 STANDARD_NAMES_VARIABLES as SNV)

results_logger_sampler = ResultsLogger(**logger_kwargs_sampler)

df_noiseless = results_logger_sampler.read_results(data_type=SNDT.OptimizationOverview,
                                                   experiment_instance_ids=['IdealSamplingQISKIT'])
df_noiseless_direct = optimization_results_noiseless.trials_dataframe

#the same logger can be used; instance_ids are used for default writing, but reading is more flexible.
#Both writing and reading filter by experiment_set_id by default
df_noisy = results_logger_sampler.read_results(data_type=SNDT.OptimizationOverview,
                                               experiment_instance_ids=['NoisySamplingQISKIT'])
df_noisy_direct = optimization_results_noisy.trials_dataframe

display(df_noiseless)
display(df_noiseless_direct)
display(df_noisy)
display(df_noisy_direct)


### Visualization

* After running optimization, we can visualize it using various methods with plotly.
* The cell below adds one column per angle to the two stored tables, then draws the two
  trajectories side by side, colored by the mean energy of each call.
* **What to change here**: `titles`, `colormap_name` and `global_normalization` (`True` puts
  both panels on one color scale).
* **Result**: the figure, written to `../temp/trajectories_plot.html`.


In [ ]:
import plotly
from quapopt.data_analysis.visualization import optimization_visualization as opt_vis

plotly.io.templates.default = "plotly"
plotly.offline.init_notebook_mode(connected=True)

x_name = f"{SNV.Angles.id_long}-0"
y_name = f"{SNV.Angles.id_long}-1"
fom_name = f"{SNV.EnergyMean.id_long}"

for _df in [df_noiseless, df_noisy]:
    _df[x_name] = _df['Angles'].apply(lambda x: x[0])
    _df[y_name] = _df['Angles'].apply(lambda x: x[1])

trajectory_fig = opt_vis.plot_multiple_heatmaps(dataframes=[df_noiseless, df_noisy],
                                                x_name=x_name,
                                                y_name=y_name,
                                                fom_name=fom_name,
                                                bounds_x=(-np.pi, np.pi),
                                                bounds_y=(-np.pi, np.pi),
                                                add_trajectories=True,
                                                titles=['Ideal', 'Noisy'],
                                                global_normalization=True,
                                                colormap_name='Viridis',
                                                minimization=True)

plotly.offline.plot(trajectory_fig,
                    filename='../temp/trajectories_plot.html')
# trajectory_fig.show()


### Grid sampling
* We can also run a simple grid sampling to get a feel for the landscape and to compare it with the optimization results.
* Here we run bruteforce grid sampling that is not scalable for higher ddepth
* We will use it to plot the background grid of points for the trajectories of the optimizer.
* Note that this works only for p=1, otherwise it's hard to visualize
* `QAOARunnerExpValuesPauliBackprop` computes the exact p=1 expectation value at each grid
  point, with no sampling.
* **What to change here**: `grid_size_total`, the total number of grid points. The grid
  splits it evenly over the two angles, so `10 ** 4` gives 100 x 100 points.
* **Result**: `grid_sampler` and `qaoa_optimizer_exp_values`. Nothing is evaluated yet.


In [ ]:
from quapopt.optimization.parameter_setting.non_adaptive_optimization.SimpleGridOptimizer import SimpleGridOptimizer
from quapopt.optimization.parameter_setting import ParametersBoundType as PBT
from quapopt.optimization.QAOA.simulation.QAOARunnerExpValuesPauliBackprop import QAOARunnerExpValuesPauliBackprop

grid_size_total = 10 ** 4

grid_sampler = SimpleGridOptimizer(parameter_bounds=[(PBT.RANGE, (-np.pi, np.pi)),
                                                     (PBT.RANGE, (-np.pi/2, np.pi/2))
                                                     ],
                                   max_trials=grid_size_total)

logger_kwargs_grid = {**logger_kwargs_main,
                      **{'experiment_instance_id': 'IdealExpValuesGrid'}}

#runner that calculates expected values for p=1 QAOA (NO SAMPLING)
qaoa_exp_values_simulator = QAOARunnerExpValuesPauliBackprop(
    hamiltonian_representations_cost=[cost_hamiltonian],
    store_full_information_in_history=False,
    simulator_name=None,
    logger_kwargs=logger_kwargs_grid,
    logging_level=None)

qaoa_optimizer_exp_values = QAOAOptimizationRunner(qaoa_runner=qaoa_exp_values_simulator)


### Evaluate the grid, without and with noise

* The same grid is evaluated twice: once without noise, and once with the measurement noise
  `CMNS` of Run 2. The exact value under noise can be computed only for uncorrelated
  classical measurement noise, which is what `CMNS` is.
* **What to change here**: `memory_intensive` (`True` reuses intermediate results across the
  grid points that share a phase angle: faster, but it takes more memory on large instances).
* **Result**: `opt_results_grid_noiseless` and `opt_results_grid_noisy`, whose
  `trials_dataframe` holds one row per grid point.


In [ ]:
#noiseless grid
best_result_grid_noiseless, opt_results_grid_noiseless = qaoa_optimizer_exp_values.run_optimization(
    qaoa_depth=1,
    number_of_function_calls=grid_size_total,
    classical_optimizer=grid_sampler,
    measurement_noise=None,
    memory_intensive=True, #for grid sampling, if True, this scales (almost) only with the number of distinct gamma angles by storing some intermediate results; but takes some memory for large system sizes
    show_progress_bar=True,
    verbosity=1,
    # analytical_betas=True
)

#noisy grid (note: this is possible to calculate only for a simple uncorrelated classical measurement noise model)
best_result_grid_noisy, opt_results_grid_noisy = qaoa_optimizer_exp_values.run_optimization(qaoa_depth=1,
                                                                                            number_of_function_calls=grid_size_total,
                                                                                            classical_optimizer=grid_sampler,
                                                                                            memory_intensive=True,
                                                                                            verbosity=1,
                                                                                            measurement_noise=CMNS,
                                                                                            show_progress_bar=True
                                                                                            )

### Draw the trajectories over the grid landscape

* The grid values are interpolated into a heatmap, and the trajectories of Run 1 and Run 2 are
  drawn on top. The loop draws the figure twice: flat, and as a 3D surface.
* **What to change here**: `show_plots_in_jupyter_cell` (`True` also shows the figures in the
  notebook, which is slower), `colormap_name` and `global_normalization`.
* **Result**: `../temp/trajectories_grid_plot_3d-False.html` and
  `../temp/trajectories_grid_plot_3d-True.html`.


In [ ]:
#this can made it a bit slower than just opening in browser, but it allows to show the plots in Jupyter cells
show_plots_in_jupyter_cell = False

names_map = {f"FunctionValue": f"{SNV.EnergyMean.id_long}"}
df_noiseless_grid = opt_results_grid_noiseless.trials_dataframe
df_noiseless_grid.rename(columns=names_map, inplace=True)
df_noisy_grid = opt_results_grid_noisy.trials_dataframe
df_noisy_grid.rename(columns=names_map, inplace=True)
for _df in [df_noiseless_grid, df_noisy_grid]:
    if x_name not in _df.columns:
        _df[x_name] = _df['Arguments'].apply(lambda x:x[0])
    if y_name not in _df.columns:
        _df[y_name] = _df['Arguments'].apply(lambda x:x[1])

for plot_in_3d in [False, True]:
    heatmap_grid_noiseless, scatter_grid_noiseless = opt_vis.get_interpolated_heatmap(df=df_noiseless_grid,
                                                                                      x_name=x_name,
                                                                                      y_name=y_name,
                                                                                      fom_name=fom_name,
                                                                                      bounds_x=(-np.pi, np.pi),
                                                                                      bounds_y=(-np.pi, np.pi),
                                                                                      in_3d=plot_in_3d
                                                                                      )
    heatmap_grid_noisy, scatter_grid_noisy = opt_vis.get_interpolated_heatmap(df=df_noisy_grid,
                                                                              x_name=x_name,
                                                                              y_name=y_name,
                                                                              fom_name=fom_name,
                                                                              bounds_x=(-np.pi, np.pi),
                                                                              bounds_y=(-np.pi, np.pi),
                                                                              in_3d=plot_in_3d
                                                                              )

    trajectory_fig2 = opt_vis.plot_multiple_heatmaps(dataframes=[df_noiseless, df_noisy],
                                                     x_name=x_name,
                                                     y_name=y_name,
                                                     fom_name=fom_name,
                                                     bounds_x=(-np.pi, np.pi),
                                                     bounds_y=(-np.pi, np.pi),
                                                     heatmaps_input=[heatmap_grid_noiseless, heatmap_grid_noisy],
                                                     scatter_inputs=[scatter_grid_noiseless, scatter_grid_noisy],
                                                     add_trajectories=True,
                                                     titles=['Ideal', 'Noisy'],
                                                     suffixes=['(Ideal)', '(Noisy)'],
                                                     global_normalization=False,
                                                     colormap_name='Viridis',
                                                     minimization=True,
                                                     in_3d=plot_in_3d)

    plotly.offline.plot(trajectory_fig2,
                        filename=f'../temp/trajectories_grid_plot_3d-{plot_in_3d}.html')
    if not show_plots_in_jupyter_cell:
        continue
    trajectory_fig2.show()
